In [ ]:
!pip install fastapi uvicorn transformers torch sentencepiece
!pip install pyngrok

In [ ]:
import os
import time
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
import uvicorn
import threading
from pyngrok import ngrok

In [ ]:
class EmbedRequest(BaseModel):
    texts: List[str]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-large")
model = AutoModel.from_pretrained("intfloat/multilingual-e5-large").to(device)
model.eval()

def average_pool(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(
        ~attention_mask[..., None].bool(), 0.0
    )
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

### Routes

In [ ]:
app = FastAPI()

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/embed")
def embed(request: EmbedRequest):

    all_embeddings = []

    batch_size = 32  # safe for T4

    for i in range(0, len(request.texts), batch_size):

        batch = request.texts[i:i+batch_size]
        batch = ["passage: " + t for t in batch]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoded)

        embeddings = average_pool(
            outputs.last_hidden_state,
            encoded["attention_mask"]
        )

        embeddings = F.normalize(embeddings, p=2, dim=1)

        all_embeddings.extend(embeddings.cpu().tolist())

    return {"embeddings": all_embeddings}

In [ ]:
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()

---

## Ngrok Tunnel

In [ ]:
os.environ["NGROK_AUTHTOKEN"] = "39I7VC4uhq8ygeXGxH7yU0TR8yU_2gNooCY8CdU5fP89JH5FT"
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

ngrok.kill()
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

In [ ]:
!pkill -f ngrok
!pkill -f uvicorn

In [ ]:
def keep_alive():
    while True:
        print("Heartbeat...")
        time.sleep(60)

keep_alive()


---



## Cloudflared Tunnel

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!cloudflared tunnel --url http://localhost:8000

## Tests

---

In [ ]:
import requests
requests.get("http://localhost:8000/health").json()

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

r = requests.get(
    "https://inadvertently-measureless-ester.ngrok-free.dev/health",
    headers=headers
)

print(r.status_code)

In [ ]:
# requests.post(
#     "https://inadvertently-measureless-ester.ngrok-free.dev/embed",
#     json={"texts": ["hello world"]}
# ).json()